El script debe realizar un agrupamiento o clustering con K-means
Consignas:

    1. Seleccionar las variables numéricas del dataset.

    2. Escalar si es necesario (normalizar)

    3. Aplicar K-means con distintos valores de k.

    4. Determinar el número óptimo de clusters (Silhouette y/o Davies-Bouldin)

**(fnlwgt excluido - es peso de muestreo, no característica)**

### Variables numericas:
 - age 
 - education-num 
 - capital-gain 
 - capital-loss 
 - hours-per-week

## Paso 1: Carga de datos y selección de variables numéricas


In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# Cargar dataset
df = cargar_dataset_adult()

# Seleccionar SOLO variables numéricas descriptivas (sin fnlwgt)
numeric_cols = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

# Extraer variables numéricas
X = df[numeric_cols].copy()

print("="*60)
print("SELECCIÓN DE VARIABLES NUMÉRICAS PARA K-MEANS")
print("="*60)
print(f"\nVariables seleccionadas: {numeric_cols}")
print(f"\nDimensión del dataset: {X.shape}")
print(f"\nEstadísticas descriptivas:\n")
print(X.describe())


SELECCIÓN DE VARIABLES NUMÉRICAS PARA K-MEANS

Variables seleccionadas: ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Dimensión del dataset: (48842, 5)

Estadísticas descriptivas:

              age  education-num  capital-gain  capital-loss  hours-per-week
count  48842.0000     48842.0000    48842.0000    48842.0000      48842.0000
mean      38.6436        10.0781     1079.0676       87.5023         40.4224
std       13.7105         2.5710     7452.0191      403.0046         12.3914
min       17.0000         1.0000        0.0000        0.0000          1.0000
25%       28.0000         9.0000        0.0000        0.0000         40.0000
50%       37.0000        10.0000        0.0000        0.0000         40.0000
75%       48.0000        12.0000        0.0000        0.0000         45.0000
max       90.0000        16.0000    99999.0000     4356.0000         99.0000


## Paso 2: Normalización/Escalado

K-means es sensible a la escala de las variables, por lo que es necesario normalizar los datos usando **StandardScaler** (Z-score normalization).


In [2]:
# Normalizar usando StandardScaler (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertir a DataFrame para mejor visualización
X_scaled_df = pd.DataFrame(X_scaled, columns=numeric_cols, index=X.index)

print("="*60)
print("NORMALIZACIÓN DE DATOS (StandardScaler)")
print("="*60)
print("\nEstadísticas ANTES de normalizar:")
print(X.describe().loc[['mean', 'std']])

print("\n\nEstadísticas DESPUÉS de normalizar:")
print(X_scaled_df.describe().loc[['mean', 'std']])




NORMALIZACIÓN DE DATOS (StandardScaler)

Estadísticas ANTES de normalizar:
          age  education-num  capital-gain  capital-loss  hours-per-week
mean  38.6436        10.0781     1079.0676       87.5023         40.4224
std   13.7105         2.5710     7452.0191      403.0046         12.3914


Estadísticas DESPUÉS de normalizar:
             age  education-num  capital-gain  capital-loss  hours-per-week
mean  2.2869e-16    -9.1360e-17    1.7603e-17   -7.5648e-18      4.2625e-17
std   1.0000e+00     1.0000e+00    1.0000e+00    1.0000e+00      1.0000e+00


## Paso 3: Aplicar K-means con distintos valores de k

Probamos K-means con k desde 2 hasta 10 clusters.


In [ ]:
# Rango de k a probar
k_range = range(2, 11)

# Listas para almacenar resultados
inertias = []
silhouette_scores = []
davies_bouldin_scores = []
kmeans_models = {}

print("="*60)
print("APLICANDO K-MEANS CON DISTINTOS VALORES DE K")
print("="*60)
print("\nEntrenando modelos...\n")

# Aplicar K-means para cada valor de k
for k in k_range:
    # Entrenar K-means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    # Guardar el modelo
    kmeans_models[k] = kmeans
    
    # Calcular métricas
    inertia = kmeans.inertia_
    silhouette = silhouette_score(X_scaled, labels)
    davies_bouldin = davies_bouldin_score(X_scaled, labels)
    
    # Almacenar resultados
    inertias.append(inertia)
    silhouette_scores.append(silhouette)
    davies_bouldin_scores.append(davies_bouldin)
    
    print(f"k={k:2d} | Inertia: {inertia:12.2f} | Silhouette: {silhouette:.4f} | Davies-Bouldin: {davies_bouldin:.4f}")

print("\n✓ Entrenamiento completado para todos los valores de k")


APLICANDO K-MEANS CON DISTINTOS VALORES DE K

Entrenando modelos...

k= 2 | Inertia:    196248.64 | Silhouette: 0.5686 | Davies-Bouldin: 0.6970
k= 3 | Inertia:    162134.90 | Silhouette: 0.2484 | Davies-Bouldin: 1.3279
k= 4 | Inertia:    119304.85 | Silhouette: 0.2598 | Davies-Bouldin: 1.0090
k= 5 | Inertia:     94490.43 | Silhouette: 0.2950 | Davies-Bouldin: 0.9170
k= 6 | Inertia:     82051.03 | Silhouette: 0.3042 | Davies-Bouldin: 0.8888


## Paso 4: Determinar el número óptimo de clusters

Usamos dos métodos para determinar el k óptimo:

1. **Coeficiente de Silhouette**: Maximizar (valores cercanos a 1 son mejores)
2. **Índice Davies-Bouldin**: Minimizar (valores menores indican mejores clusters)


In [ ]:
# Determinar k óptimos según cada métrica
max_silhouette_k = k_range[np.argmax(silhouette_scores)]
max_silhouette = max(silhouette_scores)

min_db_k = k_range[np.argmin(davies_bouldin_scores)]
min_db = min(davies_bouldin_scores)

# Resumen de resultados
print("="*60)
print("DETERMINACIÓN DEL NÚMERO ÓPTIMO DE CLUSTERS")
print("="*60)
print(f"\n Según Coeficiente de Silhouette:")
print(f"   k óptimo = {max_silhouette_k} (Silhouette = {max_silhouette:.4f})")

print(f"\n Según Índice Davies-Bouldin:")
print(f"   k óptimo = {min_db_k} (Davies-Bouldin = {min_db:.4f})")

# Tabla resumen
results_df = pd.DataFrame({
    'k': list(k_range),
    'Silhouette': silhouette_scores,
    'Davies-Bouldin': davies_bouldin_scores
})

print(f"\n\nTabla Resumen de Métricas:")
print(results_df.to_string(index=False))


## Paso 5: Análisis del modelo óptimo

Analizamos los clusters usando el k óptimo determinado por las métricas.


In [ ]:
# Usar el k óptimo según Silhouette (puedes cambiar a min_db_k si prefieres Davies-Bouldin)
k_optimal = max_silhouette_k
kmeans_optimal = kmeans_models[k_optimal]
labels_optimal = kmeans_optimal.labels_

# Agregar labels al DataFrame original
df_clustered = df[numeric_cols].copy()
df_clustered['Cluster'] = labels_optimal

print("="*60)
print(f"ANÁLISIS DEL MODELO ÓPTIMO (k={k_optimal})")
print("="*60)

# Estadísticas por cluster
print(f"\n Distribución de observaciones por cluster:")
cluster_counts = pd.Series(labels_optimal).value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(labels_optimal)) * 100
    print(f"   Cluster {cluster_id}: {count:6d} observaciones ({percentage:5.2f}%)")

# Características promedio por cluster
print(f"\n\n Características promedio por cluster:\n")
cluster_summary = df_clustered.groupby('Cluster')[numeric_cols].mean()
print(cluster_summary.round(2))

print(f"\n\n✓ Análisis completado con k óptimo = {k_optimal}")


## Paso 6: Interpretación de características por cluster

Comparación de las características promedio de cada cluster con la media global.


In [ ]:
# Interpretación textual
print("="*60)
print("INTERPRETACIÓN DE CLUSTERS")
print("="*60)

for cluster_id in sorted(cluster_summary.index):
    print(f"\n🔹 CLUSTER {cluster_id}:")
    print(f"   Tamaño: {cluster_counts[cluster_id]} observaciones ({cluster_counts[cluster_id]/len(labels_optimal)*100:.1f}%)")
    print(f"\n   Características:")
    for col in numeric_cols:
        val = cluster_summary.loc[cluster_id, col]
        global_mean = df[col].mean()
        diff_percent = ((val - global_mean) / global_mean * 100) if global_mean != 0 else 0
        
        if abs(diff_percent) > 20:  # Diferencia significativa
            direction = "↑" if diff_percent > 0 else "↓"
            print(f"   • {col}: {val:.1f} ({direction} {abs(diff_percent):.0f}% vs media global)")
        else:
            print(f"   • {col}: {val:.1f} (~ media global)")

print("\n" + "="*60)
